# 🧠 ARC-Jeni-2026 — Hybrid DSA + Program Synthesis + Chakra-Ensemble LLM

**Competition:** ARC Prize 2026 – ARC-AGI-2  
**Author:** joyjeni  
**Repo:** https://github.com/joyjeni/arc-jeni-2026  

## Architecture (3-tier)
| Tier | Method | Speed | Coverage |
|------|--------|-------|----------|
| 1 | DSA + Vedic deterministic solvers | ~60 s | 5–10% |
| 2 | LLM Program Synthesis (write+run Python) | ~30 min | 15–25% |
| 3 | Chakra-8 Ensemble direct prediction | remaining budget | 10–20% |

**Model:** Qwen3-4B fine-tuned on ARC grids (`qwen3_4b_grids15_sft139`)  
**Novel contributions:** Vedic-DSA pre-filter · Program synthesis loop · Chakra multi-prompt ensemble · 8-view augmentation voting


In [ ]:
# ── Setup & Imports ──────────────────────────────────────────────────────────
import os, sys, time, json, logging, subprocess
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('arc_jeni')

IS_KAGGLE = Path('/kaggle').exists()
IS_RERUN  = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))

DATA_DIR   = Path('/kaggle/input/arc-prize-2026-arc-agi-2') if IS_KAGGLE else Path('data')
MODEL_DIR  = Path('/kaggle/input/qwen3-4b-grids15-sft139/transformers/sft139/1') if IS_KAGGLE else None
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('.')

T0 = time.time()
TIME_LIMIT_S = 12 * 3600 if IS_RERUN else 600

print(f'IS_KAGGLE={IS_KAGGLE}  IS_RERUN={IS_RERUN}')
print(f'DATA_DIR={DATA_DIR}  MODEL_DIR={MODEL_DIR}')
print(f'TIME_LIMIT={TIME_LIMIT_S/3600:.1f}h')

In [ ]:
# ── Install / clone repo ─────────────────────────────────────────────────────
REPO_URL = 'https://github.com/joyjeni/arc-jeni-2026.git'
REPO_DIR = Path('/kaggle/working/arc-jeni-2026') if IS_KAGGLE else Path('.')

if IS_KAGGLE and not (REPO_DIR / 'src').exists():
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, str(REPO_DIR)], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo ready at', REPO_DIR)

In [ ]:
# ── Load challenge data ──────────────────────────────────────────────────────
from src.pipeline import load_challenges, load_solutions

CHALLENGES_FILE = DATA_DIR / 'arc-agi_test-challenges.json'
if not CHALLENGES_FILE.exists():
    # Fallback to evaluation set during dev
    CHALLENGES_FILE = DATA_DIR / 'arc-agi_evaluation-challenges.json'
    logger.warning('Using evaluation challenges as dev fallback')

tasks = load_challenges(CHALLENGES_FILE)
print(f'Loaded {len(tasks)} tasks from {CHALLENGES_FILE.name}')
print('Sample task IDs:', list(tasks.keys())[:5])

In [ ]:
# ── Tier 1: DSA + Vedic deterministic solvers ─────────────────────────────────
from src.solvers import REGISTRY
from src.pipeline import run_tier1

print(f'Registered solvers ({len(REGISTRY.names())}): {REGISTRY.names()}')

t1_results = run_tier1(tasks, set(), REGISTRY)
solved_ids = set(t1_results.keys())
pct = 100 * len(solved_ids) / len(tasks)
print(f'\nTier-1 solved: {len(solved_ids)}/{len(tasks)} ({pct:.1f}%)')

In [ ]:
# ── LLM Setup: Qwen3-4B from Kaggle model hub ────────────────────────────────
from src.llm.engine import build_engine

model_path = str(MODEL_DIR) if (MODEL_DIR and MODEL_DIR.exists()) else None
if model_path is None:
    logger.warning('Model dir not found — using mock backend (Tier-2/3 will be no-ops in dev)')

llm_engine = build_engine(model_path=model_path)
print('LLM engine ready:', type(llm_engine._backend).__name__)

In [ ]:
# ── Tier 2: Program Synthesis ─────────────────────────────────────────────────
from src.pipeline import run_tier2

elapsed = time.time() - T0
budget2 = min((TIME_LIMIT_S - elapsed - 900) * 0.35, 2400.0)
print(f'Tier-2 budget: {budget2:.0f}s')

t2_results = run_tier2(tasks, solved_ids, llm_engine, time_budget_s=budget2)
solved_ids |= set(t2_results.keys())
print(f'After Tier-2: {len(solved_ids)}/{len(tasks)} ({100*len(solved_ids)/len(tasks):.1f}%)')

In [ ]:
# ── Tier 3: Chakra Ensemble ───────────────────────────────────────────────────
from src.pipeline import run_tier3

elapsed = time.time() - T0
budget3 = max(0, TIME_LIMIT_S - elapsed - 900)
print(f'Tier-3 budget: {budget3:.0f}s ({budget3/3600:.2f}h)')

t3_results = run_tier3(tasks, solved_ids, llm_engine, time_budget_s=budget3, n_templates=4)
solved_ids |= set(t3_results.keys())
print(f'After Tier-3: {len(solved_ids)}/{len(tasks)} ({100*len(solved_ids)/len(tasks):.1f}%)')

In [ ]:
# ── Build & validate submission ───────────────────────────────────────────────
from src.core.submission import build_submission, validate_submission, write_submission

all_results = {**t1_results, **t2_results, **t3_results}
submission = build_submission(tasks, all_results, fallback_input=True)

errors = validate_submission(submission, tasks)
if errors:
    for e in errors[:10]:
        print('VALIDATION ERROR:', e)
else:
    print(f'✅ Submission valid: {len(submission)} tasks')

# Show sample entry
sample_tid = list(submission.keys())[0]
sample_pred = submission[sample_tid][0]
print(f'\nSample [{sample_tid}]:')
print('  attempt_1 dims:', len(sample_pred["attempt_1"]), 'x', len(sample_pred["attempt_1"][0]))
print('  attempt_2 dims:', len(sample_pred["attempt_2"]), 'x', len(sample_pred["attempt_2"][0]))

In [ ]:
# ── Write submission.json ─────────────────────────────────────────────────────
OUT_PATH = OUTPUT_DIR / 'submission.json'
write_submission(submission, OUT_PATH, validate=False)

# Verify written file
loaded = json.loads(OUT_PATH.read_text())
assert len(loaded) == len(tasks), f'Task count mismatch: {len(loaded)} vs {len(tasks)}'
for tid, preds in loaded.items():
    for pred in preds:
        assert 'attempt_1' in pred and 'attempt_2' in pred, f'Missing attempt keys in {tid}'

total_elapsed = time.time() - T0
tier_counts = {
    1: sum(1 for r in all_results.values() if r.tier == 1),
    2: sum(1 for r in all_results.values() if r.tier == 2),
    3: sum(1 for r in all_results.values() if r.tier == 3),
}
fallback = len(tasks) - len(all_results)

print(f'\n✅ submission.json written: {OUT_PATH}')
print(f'   Total tasks: {len(tasks)}')
print(f'   Tier-1 (DSA):        {tier_counts[1]}')
print(f'   Tier-2 (ProgSynth):  {tier_counts[2]}')
print(f'   Tier-3 (Chakra LLM): {tier_counts[3]}')
print(f'   Fallback (input copy): {fallback}')
print(f'   Elapsed: {total_elapsed/60:.1f} min')